In [53]:
from pathlib import Path
from functools import lru_cache
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import accelforge as af
import functools

import os
af.set_n_parallel_jobs(os.cpu_count(), print_message=True)

Using 8 parallel jobs


In [54]:
ARCH = Path("designs/arch.yaml")
WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")
MAPPING = Path("designs/gpt3_6.7B_kv_cache_mapping.yaml")

In [55]:
TARGET_GENERATED_TOKENS = 32

MAPPER_JOBS = 1

lookaheads_to_test = list(range(4, 132, 4))
token_sweep = list(range(0, 4096 + 128, 512))
output_path = Path("milestone_2_results.csv")

def batch_size_for_lookahead(lookahead: int) -> int:
    # Use one query vector per speculative position under test.
    return max(1, int(lookahead))

af.set_n_parallel_jobs(MAPPER_JOBS)

In [59]:
@functools.cache
def draft_spec(tokens: int):
    spec = af.Spec.from_yaml(
        ARCH,
        WORKLOAD,
        jinja_parse_data = {
            "BATCH_SIZE": 1,
            "N_TOKENS": tokens,
            "N_NEW_TOKENS": 1
        }
    )
    # we should ask Michael about this (should we even fuse this in the first place)
    # it might also be a good enough approximation to just the draft spec for the entire lookahead
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            print(f'Keeping all tensors in {node.name}')
            node.tensors.keep = "All"
            break
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    result = spec.map_workload_to_arch()
    return result

@functools.cache
def validator_spec(tokens: int, lookahead: int):
    spec = af.Spec.from_yaml(
        ARCH,
        VALIDATOR,
        jinja_parse_data = {
            "BATCH_SIZE": lookahead,
            "N_TOKENS": tokens + lookahead,
            "N_NEW_TOKENS": 1
        }
    )
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            node.tensors.keep = "All"
            break
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    result = spec.map_workload_to_arch()
    return result


@functools.cache
def true_validator_spec(tokens: int, lookahead: int):
    spec = af.Spec.from_yaml(
        ARCH,
        VALIDATOR,
        jinja_parse_data = {
            "BATCH_SIZE": lookahead,
            "N_TOKENS": tokens + lookahead,
            "N_NEW_TOKENS": lookahead
        }
    )
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            node.tensors.keep = "All"
            break
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    result = spec.map_workload_to_arch()
    return result

In [60]:
ARCH = Path("designs/arch.yaml")
WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")
MAPPING = Path("designs/gpt3_6.7B_kv_cache_mapping.yaml")

BATCH_SIZE = 1
N_TOKENS = 8192
N_NEW_TOKENS = 1
MAPPER_JOBS = 1

def evaluate_workload(tokens: int, lookahead: int):
    total_energy = 0.0
    total_latency = 0.0
    for i in range(lookahead):
        print(f"Currently validating {i}")
        result = draft_spec(tokens + i)
        total_energy += float(result.energy())
        total_latency += float(result.latency())

    print("on Validator")
    result = validator_spec(tokens, lookahead)
    total_energy += float(result.energy())
    total_latency += float(result.latency())

    return total_energy, total_latency

def default_kv_cached(tokens: int, lookahead: int):
    total_energy = 0.0
    total_latency = 0.0
    for i in range(lookahead):
        print(f"Iteration {i}")
        result = validator_spec(tokens + i, 1)
        total_energy += float(result.energy())
        total_latency += float(result.latency())

    # return per-token cost
    return total_energy / lookahead, total_latency / lookahead


In [61]:
def expected_tokens_per_round(gamma: int, alpha: float) -> float:
    if alpha >= 1.0:
        return gamma + 1
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)

def speculative_decoding(context_len: int, gamma: int, alpha: float):
    round_energy = 0.0
    round_latency = 0.0

    for i in range(gamma):
        result = draft_spec(context_len + i)
        round_energy += float(result.energy())
        round_latency += float(result.latency())

    result = true_validator_spec(context_len, gamma)
    round_energy += float(result.energy())
    round_latency += float(result.latency())

    expected_yield = expected_tokens_per_round(gamma, alpha)

    return round_energy / expected_yield, round_latency / expected_yield


In [ ]:
context_lengths = list(range(128, 129, 128))
gammas = [3, 6, 9]
results = []

for ctx in context_lengths:
    for gamma in gammas:
        print(f"Context={ctx}, gamma={gamma}: running baseline...")
        base_energy_per_tok, base_latency_per_tok = default_kv_cached(ctx, gamma)

        print(f"  running speculative decoding...")
        spec_energy_per_tok, spec_latency_per_tok = speculative_decoding(ctx, gamma, ALPHA)
        results.append({
            "context_len": ctx,
            "gamma": gamma,
            "alpha": ALPHA,
            "base_energy_per_tok": base_energy_per_tok,
            "base_latency_per_tok": base_latency_per_tok,
            "spec_energy_per_tok": spec_energy_per_tok,
            "spec_latency_per_tok": spec_latency_per_tok,
            "latency_speedup": base_latency_per_tok / spec_latency_per_tok,
            "energy_ratio": spec_energy_per_tok / base_energy_per_tok,
        })

df = pd.DataFrame(results)
df.to_csv("spec_decoding_results_1.csv", index=False)
print("Saved to spec_decoding_results_1.csv")
df


In [ ]:
context_lengths = list(range(128, 1024, 128))
gammas = [3, 6, 9]
results = []

for ctx in context_lengths:
    for gamma in gammas:
        print(f"Context={ctx}, gamma={gamma}: running baseline...")
        base_energy_per_tok, base_latency_per_tok = default_kv_cached(ctx, gamma)

        print(f"  running speculative decoding...")
        spec_energy_per_tok, spec_latency_per_tok = speculative_decoding(ctx, gamma, ALPHA)
        results.append({
            "context_len": ctx,
            "gamma": gamma,
            "alpha": ALPHA,
            "base_energy_per_tok": base_energy_per_tok,
            "base_latency_per_tok": base_latency_per_tok,
            "spec_energy_per_tok": spec_energy_per_tok,
            "spec_latency_per_tok": spec_latency_per_tok,
            "latency_speedup": base_latency_per_tok / spec_latency_per_tok,
            "energy_ratio": spec_energy_per_tok / base_energy_per_tok,
        })
        pd.DataFrame(results).to_csv("spec_decoding_results2.csv", index=False)

df2 = pd.DataFrame(results)
print("Saved to spec_decoding_results2.csv")
df2
